# Lia — LoRA emocional no Qwen Base, inferência ICL preservada

Este é o experimento de **adaptação real do TTS**. Treina LoRA no Qwen3-TTS Base com instruções de emoção explícitas, porque o trainer anterior ignorava `emotion_audit`. Na inferência, o adapter continua no modo Base e recebe a referência ICL original (`ref_audio` + transcrição); **não** converte o modelo para `custom_voice`.

O objetivo é separar: (a) identidade vocal ancorada pelo ICL Base; (b) efeito da instrução no TTS; (c) alteração adicional produzida pelo adapter emocional. Smoke de um update vem antes do treino completo. FP32 + SDPA + batch 1; sem A100. Isso é experimental; o resultado ainda precisa passar avaliação auditiva.

Usa o ZIP curado já baixado em `/content/lia_qwen_lora`; não reenvia o WAV original. Dados e saídas permanecem privados no Colab.


In [ ]:
# 0. Dependências auxiliares isoladas; mantém Torch/Torchaudio CUDA do Colab
from pathlib import Path
import os, sys, subprocess, json, csv, shutil, zipfile, hashlib, importlib.util, gc, time
import torch
ROOT=Path('/content/lia_qwen_lora'); ROOT.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=str(ROOT/'hf_home')  # definido antes de importar huggingface_hub
DEPS=ROOT/'python_deps'; DEPS.mkdir(exist_ok=True)
packages=['transformers==4.57.3','accelerate==1.12.0','peft==0.17.1','huggingface_hub==0.36.2','safetensors==0.7.0','tokenizers==0.22.1','librosa==0.11.0','soxr==0.5.0.post1','sox==1.5.0','einops==0.8.1','onnxruntime==1.23.2']
marker=DEPS/'.lora_deps_ready'
if not marker.exists():
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--no-deps','--target',str(DEPS),*packages],check=True)
    marker.write_text('\n'.join(packages)+'\n',encoding='utf-8')
sys.path.insert(0,str(DEPS))
# Qwen é áudio; impede auto-detecção de uma torchvision incompatível não usada.
import transformers.utils.import_utils as _hf_import_utils
_hf_import_utils._torchvision_available=False
import transformers, accelerate, peft, huggingface_hub, soundfile as sf
from google.colab import files
from IPython.display import display, Audio
if not torch.cuda.is_available(): raise RuntimeError('Selecione uma GPU CUDA (T4) no runtime Colab.')
props=torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory/1024**3:.1f} GiB | BF16 reportado: {torch.cuda.is_bf16_supported()}')
print('Torch ativo:',torch.__version__,'| Transformers isolado:',transformers.__version__,'| PEFT:',peft.__version__)
print('Dados/checkpoints em',ROOT)
existing_zips=sorted(p for p in ROOT.glob('*.zip') if p.is_file())
if len(existing_zips)==1:
    bundle=existing_zips[0]
    print('Reutilizando o ZIP curado já presente:',bundle.name)
else:
    print('Selecione o ZIP curado já baixado.')
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('Envie exatamente um ZIP.')
    zip_name,zip_bytes=next(iter(uploaded.items()))
    if not zip_name.lower().endswith('.zip'): raise ValueError('Esperado um arquivo ZIP.')
    bundle=ROOT/Path(zip_name).name; bundle.write_bytes(zip_bytes); del zip_bytes,uploaded; gc.collect()
with zipfile.ZipFile(bundle) as zf:
    root_resolved=ROOT.resolve()
    for item in zf.infolist():
        target=(ROOT/item.filename).resolve()
        if target!=root_resolved and root_resolved not in target.parents: raise ValueError('Caminho inseguro no ZIP; extração bloqueada.')
    zf.extractall(ROOT)
manifest=json.loads((ROOT/'training_bundle_manifest.json').read_text(encoding='utf-8'))
if manifest.get('format')!='lia-qwen-training-bundle-v1': raise ValueError('Formato de pacote inesperado.')
for required in ('train_raw.jsonl','holdout_raw.jsonl','review.csv','review_approval.json','emotion_audit.csv','speaker_reference.wav'):
    if not (ROOT/required).is_file(): raise FileNotFoundError(f'Arquivo obrigatório ausente: {required}')
review_hash=hashlib.sha256((ROOT/'review.csv').read_bytes()).hexdigest()
approval=json.loads((ROOT/'review_approval.json').read_text(encoding='utf-8'))
if approval.get('review_csv_sha256')!=review_hash or manifest.get('review_csv_sha256')!=review_hash: raise RuntimeError('Hash da revisão não corresponde à aprovação; pacote alterado/incompleto.')
actual_train=sum(1 for line in (ROOT/'train_raw.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())
actual_holdout=sum(1 for line in (ROOT/'holdout_raw.jsonl').read_text(encoding='utf-8').splitlines() if line.strip())
if actual_train!=manifest.get('train_rows') or actual_holdout!=manifest.get('holdout_rows'):
    raise RuntimeError(f'Contagem no manifest diverge do conteúdo: train {actual_train}/{manifest.get("train_rows")}, holdout {actual_holdout}/{manifest.get("holdout_rows")}')
print('Pacote curado:',actual_train,'treino;',actual_holdout,'holdout; review hash',review_hash[:12])

## 1. Validar e limitar duração para a T4
Expande caminhos portáteis/antigos do ZIP para caminhos absolutos locais, verifica os WAVs e limita a 12 s para reduzir memória de ativações. Os originais não são alterados; holdout permanece separado.

In [ ]:
# 1. Paths locais + teto de duração (apenas para esta experiência T4)
from pathlib import Path
import soundfile as sf
MAX_CLIP_SECONDS=12.0

def resolve_bundled_audio(value):
    p=Path(value)
    if p.is_absolute() and p.is_file() and p.is_relative_to(ROOT.resolve()): return p.resolve()
    if not p.is_absolute():
        portable=(ROOT/p).resolve()
        if portable.is_file() and portable.is_relative_to(ROOT.resolve()): return portable
    # JSONL de runtime antigo pode guardar caminho absoluto; resolve pelo nome no ZIP portátil.
    matches=list(ROOT.rglob(p.name))
    matches=[x.resolve() for x in matches if x.is_file() and x.suffix.lower()=='.wav']
    if len(matches)!=1: raise FileNotFoundError(f'Não consegui resolver univocamente o áudio {value!r}: {matches[:4]}')
    return matches[0]

def prepare_split(src_name,dst_name,min_rows):
    kept=[]; dropped=[]; refs=set()
    for line_no,line in enumerate((ROOT/src_name).read_text(encoding='utf-8').splitlines(),1):
        if not line.strip(): continue
        row=json.loads(line); audio=resolve_bundled_audio(row['audio']); ref=resolve_bundled_audio(row['ref_audio'])
        if not audio.is_relative_to(ROOT.resolve()) or not ref.is_relative_to(ROOT.resolve()): raise ValueError('Áudio fora do ZIP/runtime privado.')
        info=sf.info(audio)
        if info.channels!=1 or info.samplerate!=24000: raise ValueError(f'{audio.name}: esperava mono 24 kHz, recebi {info.channels}ch/{info.samplerate}Hz')
        refs.add(ref)
        if info.duration>MAX_CLIP_SECONDS: dropped.append((audio.name,info.duration)); continue
        row['audio']=str(audio); row['ref_audio']=str(ref); kept.append(row)
    if len(refs)!=1: raise ValueError(f'{src_name}: esperado um ref_audio fixo, encontrei {len(refs)}.')
    ref_info=sf.info(next(iter(refs)))
    if ref_info.channels!=1 or ref_info.samplerate!=24000: raise ValueError(f'ref_audio: esperava mono 24 kHz, recebi {ref_info.channels}ch/{ref_info.samplerate}Hz')
    if len(kept)<min_rows: raise ValueError(f'{src_name}: restaram {len(kept)} recortes <= {MAX_CLIP_SECONDS}s; mínimo para esta divisão: {min_rows}.')
    (ROOT/dst_name).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in kept),encoding='utf-8')
    return kept,dropped
train_rows,dropped_train=prepare_split('train_raw.jsonl','train_t4.jsonl',5)
val_rows,dropped_val=prepare_split('holdout_raw.jsonl','holdout_t4.jsonl',1)
print(f'Treino: {len(train_rows)} recortes, {sum(sf.info(r["audio"]).duration for r in train_rows)/60:.1f} min; holdout: {len(val_rows)}.')
print(f'Excluídos temporariamente por >{MAX_CLIP_SECONDS}s: treino {len(dropped_train)}, holdout {len(dropped_val)}.')
print('Emoções anotadas (metadado, não condicionamento direto):',sorted({r.get('emotion_audit','nao_rotulado') for r in train_rows}))

## 2. Qwen fixado + trainer emocional
O trainer customizado injeta, no treino, os mesmos embeddings de instrução que o gerador Qwen usa. As classes sem rótulo revisado são excluídas; classes elegíveis são balanceadas por amostragem. A inferência será Base+ICL com adapter aplicado, nunca `generate_custom_voice`.


In [ ]:
# 2. Repositórios e snapshots de modelo pinados
QWEN_COMMIT='0c6a7cbb6c8421a46332f8c2434c7825c4c855ef'
COMPANION_COMMIT='079db4ba844037f970b488dc96b32e5c7490642f'
MODEL_ID='Qwen/Qwen3-TTS-12Hz-0.6B-Base'; MODEL_REV='5d83992436eae1d760afd27aff78a71d676296fc'
TOKENIZER_ID='Qwen/Qwen3-TTS-Tokenizer-12Hz'; TOKENIZER_REV='7dd38ad4e9bad454aae9cd937d0cd577604fe229'
LORA_RANK=16; LORA_ALPHA=32; TRAIN_EPOCHS=3; TRAIN_LR=2e-6
LORA_TARGET_MODULES='q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj'
QWEN_DIR=ROOT/'Qwen3-TTS'; COMP_DIR=ROOT/'qwen3-tts-lora-finetuning'
def clone_pinned(url,dest,sha):
    if not dest.exists(): subprocess.run(['git','clone','--quiet','--no-checkout',url,str(dest)],check=True)
    subprocess.run(['git','-C',str(dest),'fetch','--quiet','--depth','1','origin',sha],check=True)
    # Idempotente: remove o patch local da execução anterior antes de reaplicá-lo.
    subprocess.run(['git','-C',str(dest),'reset','--hard',sha],check=True,stdout=subprocess.DEVNULL)
    subprocess.run(['git','-C',str(dest),'clean','-fd'],check=True,stdout=subprocess.DEVNULL)
    got=subprocess.check_output(['git','-C',str(dest),'rev-parse','HEAD'],text=True).strip()
    if got!=sha: raise RuntimeError(f'Commit inesperado: {got}')
clone_pinned('https://github.com/QwenLM/Qwen3-TTS.git',QWEN_DIR,QWEN_COMMIT)
clone_pinned('https://github.com/instavar/qwen3-tts-lora-finetuning.git',COMP_DIR,COMPANION_COMMIT)
patch=COMP_DIR/'patches'/'0001-qwen3-tts-lora.patch'
subprocess.run(['git','-C',str(QWEN_DIR),'apply','--check',str(patch)],check=True)
subprocess.run(['git','-C',str(QWEN_DIR),'apply',str(patch)],check=True)
trainer=QWEN_DIR/'finetuning'/'sft_12hz_lora.py'
source=trainer.read_text(encoding='utf-8')
def replace_once(text,old,new,label):
    n=text.count(old)
    if n!=1: raise RuntimeError(f'Patch T4 abortado: {label} devia ocorrer uma vez, ocorreu {n}.')
    return text.replace(old,new,1)
source=replace_once(source,'torch_dtype=torch.bfloat16,','dtype=torch.float32,','FP32 para T4; API Transformers atual')
source=replace_once(source,'log_with="tensorboard",','log_with=None,','sem TensorBoard')
source=replace_once(source,'    config = AutoConfig.from_pretrained(MODEL_PATH)\n','''    config = AutoConfig.from_pretrained(MODEL_PATH)
    qwen3tts.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    # Desliga KV cache em todos os submodelos; cache não é necessário no SFT e custa VRAM.
    for _module in qwen3tts.model.modules():
        _config = getattr(_module, "config", None)
        if _config is not None and hasattr(_config, "use_cache"):
            _config.use_cache = False
    # Qwen3TTS é multimodal e não implementa o hook genérico do Transformers.
    # PEFT 0.17 chama enable_input_require_grads() ao detectar checkpointing;
    # exponha o embedding textual usado pelo trainer para esse hook funcionar.
    qwen3tts.model.get_input_embeddings = lambda: qwen3tts.model.talker.model.text_embedding
''','gradient checkpointing')
source=replace_once(source,'    parser.add_argument("--eval_every", type=int, default=1)\n','''    parser.add_argument("--eval_every", type=int, default=1)
    parser.add_argument("--max_train_steps", type=int, default=0)
''','limite do smoke')
source=replace_once(source,'            if step % 10 == 0:\n                accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")\n','''            if step % 10 == 0:
                accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")
            if args.max_train_steps and step + 1 >= args.max_train_steps:
                peak=torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0.0
                accelerator.print(f"Smoke test encerrou em {step+1} passo; pico CUDA alocado {peak:.2f} GiB; nenhum checkpoint salvo.")
                return
''','saída limitada do smoke')
trainer.write_text(source,encoding='utf-8')
# Baixa helper revisado do Git e verifica hash antes de treinar.
import urllib.request
EMOTION_TRAINER_SHA256='8e60970707c865d9669d1e707d8f2cb8423e1001c5d80daf32321b56a2d6cef7'
helper_url='https://raw.githubusercontent.com/BloomRX/Lia-Omni/arena/01a0ec89-lia-code/benchmark/qwen3_tts_emotion_lora.py'
emotion_trainer=ROOT/'qwen3_tts_emotion_lora.py'
with urllib.request.urlopen(helper_url,timeout=30) as _response: _helper_bytes=_response.read()
if hashlib.sha256(_helper_bytes).hexdigest()!=EMOTION_TRAINER_SHA256: raise RuntimeError('Hash do trainer emocional divergente; pare sem treinar.')
emotion_trainer.write_bytes(_helper_bytes)
prep=QWEN_DIR/'finetuning'/'prepare_data.py'; ps=prep.read_text(encoding='utf-8')
if ps.count('BATCH_INFER_NUM = 32')!=1: raise RuntimeError('Batch do tokenizer mudou; revisão necessária.')
prep.write_text(ps.replace('BATCH_INFER_NUM = 32','BATCH_INFER_NUM = 1',1),encoding='utf-8')
infer=QWEN_DIR/'finetuning'/'infer_lora_custom_voice.py'; ins=infer.read_text(encoding='utf-8')
ins=replace_once(ins,'if __name__ == \"__main__\":\n','if __name__ == \"__main__\":\n    main()\n','entrypoint do helper de inferência')
ins=replace_once(ins,'torch_dtype=dtype_map[args.dtype],','dtype=dtype_map[args.dtype],','API Transformers atual na inferência')
ins=replace_once(ins,'    parser.add_argument("--language", default="auto")\n','''    parser.add_argument("--language", default="auto")
    parser.add_argument("--instruct", default="")
''','instruct CLI')
ins=replace_once(ins,'        language=args.language,\n        max_new_tokens=args.max_new_tokens,\n','''        language=args.language,
        instruct=args.instruct or None,
        max_new_tokens=args.max_new_tokens,
''','passagem do instruct')
infer.write_text(ins,encoding='utf-8')
os.environ['HF_HOME']=str(ROOT/'hf_home')
from huggingface_hub import snapshot_download
if globals().get('model_path') and Path(model_path).is_dir():
    print('Reutilizando o snapshot local do modelo base:',model_path)
else:
    model_path=snapshot_download(MODEL_ID,revision=MODEL_REV,cache_dir=str(ROOT/'hf_home'))
if globals().get('tokenizer_path') and Path(tokenizer_path).is_dir():
    print('Reutilizando o snapshot local do tokenizer:',tokenizer_path)
else:
    tokenizer_path=snapshot_download(TOKENIZER_ID,revision=TOKENIZER_REV,cache_dir=str(ROOT/'hf_home'))
train_env=os.environ.copy(); train_env['HF_HOME']=str(ROOT/'hf_home')
(DEPS/'sitecustomize.py').write_text('try:\n import transformers.utils.import_utils as _u\n _u._torchvision_available=False\nexcept Exception:\n pass\n',encoding='utf-8')
train_env['PYTHONPATH']=os.pathsep.join([str(DEPS),str(QWEN_DIR),str(QWEN_DIR/'finetuning'),str(ROOT),train_env.get('PYTHONPATH','')])
train_env['PYTHONUNBUFFERED']='1'
print(f'Qwen pinned + trainer emocional verificado. FP32/SDPA, rank {LORA_RANK}, {TRAIN_EPOCHS} épocas; Base ICL será mantido na inferência.')

## 3. Tokenizar os códigos de áudio
Executa o tokenizer oficial sobre treino e holdout. Batch 1 para limitar a VRAM. Ainda não treina.

In [ ]:
# 3. Preparar códigos de áudio, mostrando o traceback real do subprocesso
import collections
# Primeiro valida imports/dependências sem carregar pesos; se falhar, imprime toda a mensagem.
# Qwen importa também o wrapper Python 'sox' durante o carregamento dos módulos 25 Hz.
# A rota Lia usa tokenizer 12 Hz; não é necessário compilar FlashAttention nem instalar o binário SoX.
if importlib.util.find_spec('sox') is None:
    print('Instalando o wrapper Python sox==1.5.0 no target isolado do notebook...')
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--no-deps','--target',str(DEPS),'sox==1.5.0'],check=True)
    importlib.invalidate_caches()
    sys.path.insert(0,str(DEPS))
probe_code="import torch,torchaudio,transformers,accelerate,peft,librosa,soundfile,sox; from qwen_tts import Qwen3TTSTokenizer; print('imports OK',torch.__version__,torchaudio.__version__,transformers.__version__,peft.__version__)"
probe=subprocess.run([sys.executable,'-c',probe_code],env=train_env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(probe.stdout or '(sem saída do teste de imports)')
if probe.returncode: raise RuntimeError(f'Teste de imports falhou (exit {probe.returncode}); veja o traceback acima.')

prep=QWEN_DIR/'finetuning'/'prepare_data.py'
for src,dst in [('train_t4.jsonl','train_t4_codes.jsonl'),('holdout_t4.jsonl','holdout_t4_codes.jsonl')]:
    outpath=ROOT/dst
    input_path=ROOT/src
    expected=sum(1 for line in input_path.read_text(encoding='utf-8').splitlines() if line.strip())
    if outpath.is_file():
        try:
            with outpath.open(encoding='utf-8') as f: cached=[json.loads(line) for line in f if line.strip()]
            reusable=(len(cached)==expected and all('audio_codes' in row for row in cached))
        except Exception: reusable=False
        if reusable:
            print(f'Reutilizando {dst} validado ({expected} exemplos).'); continue
        outpath.unlink()  # remove somente saída gerada inválida/incompleta
    cmd=[sys.executable,'-u',str(prep),'--device','cuda:0','--tokenizer_model_path',tokenizer_path,
         '--input_jsonl',str(input_path),'--output_jsonl',str(outpath)]
    print('\nExecutando:',src,'→',dst,flush=True)
    proc=subprocess.Popen(cmd,env=train_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    tail=collections.deque(maxlen=100)
    for line in proc.stdout:
        print(line,end='',flush=True); tail.append(line)
    rc=proc.wait()
    if rc:
        print('\n--- últimas linhas do processo de tokenização ---\n'+''.join(tail),flush=True)
        raise RuntimeError(f'prepare_data.py falhou no split {src} (exit {rc}); traceback integral exibido acima.')
    if not outpath.is_file() or outpath.stat().st_size==0: raise RuntimeError(f'Preparação incompleta: {dst}')
    with outpath.open(encoding='utf-8') as f: coded=[json.loads(line) for line in f if line.strip()]
    expected=sum(1 for line in (ROOT/src).read_text(encoding='utf-8').splitlines() if line.strip())
    if len(coded)!=expected or any('audio_codes' not in row for row in coded):
        raise RuntimeError(f'{dst}: códigos ausentes ou contagem divergente ({len(coded)}/{expected}); não prossiga ao smoke.')
print('JSONLs codificados localmente; nenhum áudio enviado a serviço.')

# Filtrar rótulos revisados para treino emocional condicionado; mantém audio_codes e holdout.
EMOTION_LABELS={'alegre','comemorativa','neutro','calma','assertiva','triste','irritada'}
def make_emotion_split(src_name,dst_name,active_labels=None):
    rows=[json.loads(line) for line in (ROOT/src_name).read_text(encoding='utf-8').splitlines() if line.strip()]
    eligible=[r for r in rows if str(r.get('emotion_audit','')).strip().lower() in EMOTION_LABELS]
    counts=collections.Counter(str(r.get('emotion_audit','')).strip().lower() for r in eligible)
    if active_labels is None: active_labels={label for label,count in counts.items() if count>=2}
    kept=[r for r in eligible if str(r.get('emotion_audit','')).strip().lower() in active_labels]
    if not kept: raise RuntimeError(f'{src_name}: nenhum exemplo rotulado. Contagens: {dict(counts)}')
    (ROOT/dst_name).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in kept),encoding='utf-8')
    return kept,counts,active_labels
emotion_train_rows,emotion_counts,active_emotions=make_emotion_split('train_t4_codes.jsonl','train_t4_emotion_codes.jsonl')
if len(active_emotions)<2 or len(emotion_train_rows)<8:
    raise RuntimeError(f'Poucos dados para adapter emocional: {len(emotion_train_rows)} exemplos, {dict(emotion_counts)}. Não iniciar treino longo.')
try:
    emotion_val_rows,_,_=make_emotion_split('holdout_t4_codes.jsonl','holdout_t4_emotion_codes.jsonl',active_emotions)
except RuntimeError as exc:
    print('Holdout sem amostras rotuladas elegíveis; validação será omitida, treino continua separado:',exc)
    emotion_val_rows=[]
print('Emoções usadas:',sorted(active_emotions),'| contagens treino:',dict(emotion_counts))
print(f'Treino emocional: {len(emotion_train_rows)}; holdout emocional: {len(emotion_val_rows)}')


## 4. Smoke: um update emocional, sem salvar adapter
Um exemplo revisado, rank/módulos iguais ao treino, um backward/update. Se falhar ou der OOM, pare; o treino longo não é iniciado automaticamente.


In [ ]:
# Smoke emocional: um optimizer update em um único exemplo e sem checkpoint
smoke_jsonl=ROOT/'train_emotion_smoke.jsonl'
smoke_jsonl.write_text(json.dumps(emotion_train_rows[0],ensure_ascii=False)+'\n',encoding='utf-8')
smoke_out=ROOT/'emotion_smoke_discarded'
if smoke_out.exists(): shutil.rmtree(smoke_out)
smoke_cmd=[sys.executable,'-u',str(emotion_trainer),'--init_model_path',model_path,'--output_model_path',str(smoke_out),
 '--train_jsonl',str(smoke_jsonl),'--batch_size','1','--lr',str(TRAIN_LR),'--num_epochs','1','--speaker_name','lia_emotional',
 '--gradient_accumulation_steps','1','--mixed_precision','no','--attn_implementation','sdpa','--lora_rank',str(LORA_RANK),
 '--lora_alpha',str(LORA_ALPHA),'--lora_dropout','0.05','--lora_target_modules',LORA_TARGET_MODULES,'--max_train_steps','1']
print('Smoke condicionado à emoção; exatamente um update, sem salvar adapter.',flush=True)
proc=subprocess.Popen(smoke_cmd,env=train_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
if proc.wait()!=0: raise RuntimeError('Smoke emocional falhou; treino completo não iniciado.')
print('Smoke emocional passou. Revise o log acima antes do treino longo.')


## 5. Treino LoRA condicionado à emoção
O dataset do trainer anterior ignorava os rótulos. Este trainer converte `emotion_audit` em instrução e a injeta no começo da sequência do TTS durante a loss, com classes balanceadas. O modelo-base continua intacto.


In [ ]:
# Treino do adapter emocional. Base congelada; só os pesos LoRA são atualizados.
RUN_EMOTION_LORA=False  # habilite somente depois de o smoke passar e seus logs serem revisados
if RUN_EMOTION_LORA:
    run_id=time.strftime('%Y%m%d_%H%M%S')+'_'+str(time.time_ns()%1_000_000_000)
    adapter_dir=ROOT/f'lia_emotion_lora_{run_id}'
    status_path=ROOT/'emotion_lora_training_success.json'
    status_path.unlink(missing_ok=True)
    cmd=[sys.executable,'-u',str(emotion_trainer),'--init_model_path',model_path,'--output_model_path',str(adapter_dir),
      '--train_jsonl',str(ROOT/'train_t4_emotion_codes.jsonl'),
      '--batch_size','1','--lr',str(TRAIN_LR),'--num_epochs',str(TRAIN_EPOCHS),'--speaker_name','lia_emotional',
      '--gradient_accumulation_steps','4','--mixed_precision','no','--attn_implementation','sdpa',
      '--lora_rank',str(LORA_RANK),'--lora_alpha',str(LORA_ALPHA),'--lora_dropout','0.05','--lora_target_modules',LORA_TARGET_MODULES]
    if emotion_val_rows: cmd += ['--val_jsonl',str(ROOT/'holdout_t4_emotion_codes.jsonl')]
    proc=subprocess.Popen(cmd,env=train_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout: print(line,end='',flush=True)
    if proc.wait()!=0: raise RuntimeError('Treino emocional falhou; adapter não será aprovado.')
    checkpoints=sorted(adapter_dir.glob('checkpoint-epoch-*'))
    if not checkpoints: raise FileNotFoundError('Treino terminou sem checkpoint LoRA.')
    adapter_path=checkpoints[-1]
    required_files=['adapter_config.json','adapter_model.safetensors','speaker_embedding.safetensors']
    missing=[name for name in required_files if not (adapter_path/name).is_file()]
    if missing: raise FileNotFoundError(f'Adapter incompleto: {missing}')
    def sha256_file(path):
        digest=hashlib.sha256()
        with Path(path).open('rb') as f:
            for block in iter(lambda:f.read(1024*1024),b''): digest.update(block)
        return digest.hexdigest()
    status={'status':'success','adapter_path':str(adapter_path),'base_model_id':MODEL_ID,'base_revision':MODEL_REV,
      'qwen_commit':QWEN_COMMIT,'trainer_sha256':EMOTION_TRAINER_SHA256,'emotion_labels':sorted(active_emotions),
      'train_codes_sha256':sha256_file(ROOT/'train_t4_emotion_codes.jsonl'),
      'speaker_reference_sha256':hashlib.sha256((ROOT/'speaker_reference.wav').read_bytes()).hexdigest(),
      'epochs':TRAIN_EPOCHS,'rank':LORA_RANK,'created_at':time.strftime('%Y-%m-%dT%H:%M:%S%z')}
    status_path.write_text(json.dumps(status,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Adapter emocional registrado:',adapter_path)
else: print('Treino emocional desativado.')


## 6. A/B do modelo adaptado com a mesma voz ICL
Mesma frase, áudio/transcrição de referência e instrução: Base vs Base+LoRA. O adapter é aplicado sem marcar o modelo como `custom_voice`; a síntese usa `generate_voice_clone` com full ICL e `instruct_ids` explícito.


In [ ]:
# A/B Base vs adapter emocional, sempre com o mesmo clone ICL e a mesma frase
from IPython.display import Audio, Markdown, display
status_path=ROOT/'emotion_lora_training_success.json'
if not status_path.is_file(): raise RuntimeError('Treino emocional não concluído; inferência bloqueada.')
status=json.loads(status_path.read_text(encoding='utf-8'))
if status.get('status')!='success' or status.get('base_revision')!=MODEL_REV or status.get('qwen_commit')!=QWEN_COMMIT:
    raise RuntimeError('Proveniência do adapter/base inválida.')
if status.get('train_codes_sha256')!=sha256_file(ROOT/'train_t4_emotion_codes.jsonl'):
    raise RuntimeError('JSONL emocional difere do usado no treino.')
if status.get('speaker_reference_sha256')!=hashlib.sha256((ROOT/'speaker_reference.wav').read_bytes()).hexdigest():
    raise RuntimeError('A referência mudou desde o treino.')
adapter_path=Path(status['adapter_path']).resolve()
if not adapter_path.is_relative_to(ROOT.resolve()): raise RuntimeError('Adapter fora do runtime privado.')
run_tag=time.strftime('%Y%m%d_%H%M%S')
with (ROOT/'review.csv').open(encoding='utf-8-sig',newline='') as f: ref_rows=[r for r in csv.DictReader(f) if r.get('reference','').strip().lower() in {'yes','true','1','sim'}]
if len(ref_rows)!=1 or not ref_rows[0].get('text','').strip(): raise RuntimeError('Falta transcrição revisada da speaker_reference.')
ref_text=ref_rows[0]['text'].strip(); ref_audio=str(ROOT/'speaker_reference.wav')
target_text='Você conseguiu! Eu sabia que era capaz. Estou muito orgulhosa de você!'
program = """import sys,json\nfrom pathlib import Path\nimport torch,soundfile as sf\nsys.path.insert(0,sys.argv[1]);sys.path.insert(0,sys.argv[2])\nimport transformers.utils.import_utils as iu\niu._torchvision_available=False\nfrom qwen_tts import Qwen3TTSModel\nfrom peft import PeftModel\nmodel_path,adapter_path,ref_audio,ref_text,target,outdir,tag=sys.argv[3:10]\noutdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\nmodel=Qwen3TTSModel.from_pretrained(model_path,device_map='cuda:0',dtype=torch.float32,attn_implementation='sdpa')\nassert model.model.tts_model_type=='base',model.model.tts_model_type\nprompts={'alegre':'Fale em português brasileiro com alegria calorosa e genuína, energia expressiva e sorriso na voz, sem gritar.','neutro':'Fale em português brasileiro de forma neutra, natural e conversacional, sem dramatizar.'}\ndef generate(label,instruct,adapted):\n    if adapted:\n        peft=PeftModel.from_pretrained(model.model,adapter_path)\n        model.model=peft.merge_and_unload()\n        assert model.model.tts_model_type=='base',model.model.tts_model_type\n    clone_prompt=model.create_voice_clone_prompt(ref_audio=ref_audio,ref_text=ref_text,x_vector_only_mode=False)\n    kwargs={}\n    if instruct:\n        kwargs['instruct_ids']=model._tokenize_texts([model._build_instruct_text(instruct)])\n    torch.manual_seed(1234)\n    wavs,sr=model.generate_voice_clone(text=target,language='Portuguese',voice_clone_prompt=clone_prompt,max_new_tokens=180,do_sample=True,**kwargs)\n    path=outdir/f'lia_{label}_{tag}.wav';sf.write(path,wavs[0],sr)\n    print(json.dumps({'label':label,'path':str(path),'seconds':len(wavs[0])/sr,'mode':'base_icl_plus_emotion_instruction','adapter':adapted},ensure_ascii=False),flush=True)\ngenerate('base_icl_neutral',None,False)\ngenerate('base_icl_alegre',prompts['alegre'],False)\ngenerate('adapter_icl_alegre',prompts['alegre'],True)\ngenerate('adapter_icl_neutro',prompts['neutro'],True)\n"""
script_path=ROOT/'run_emotion_adapter_icl.py';script_path.write_text(program,encoding='utf-8')
outdir=ROOT/'emotion_adapter_icl_outputs';outdir.mkdir(exist_ok=True)
env=os.environ.copy();env['HF_HOME']=str(ROOT/'hf_home');env['PYTHONUNBUFFERED']='1';env['USE_TF']='0';env['USE_FLAX']='0'
env['PYTHONPATH']=os.pathsep.join([str(DEPS),str(QWEN_DIR),str(QWEN_DIR/'finetuning'),str(ROOT),env.get('PYTHONPATH','')])
cmd=[sys.executable,'-u',str(script_path),str(DEPS),str(QWEN_DIR),str(model_path),str(adapter_path),ref_audio,ref_text,target_text,str(outdir),run_tag]
proc=subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
if proc.wait()!=0: raise RuntimeError('A/B Base ICL + adapter falhou; consulte o traceback acima.')
for label,title in [('base_icl_neutral','Base ICL sem instrução'),('base_icl_alegre','Base ICL + alegre'),('adapter_icl_alegre','Adapter emocional + ICL + alegre'),('adapter_icl_neutro','Adapter emocional + ICL + neutro')]:
    path=outdir/f'lia_{label}_{run_tag}.wav'
    if not path.is_file(): raise FileNotFoundError(path)
    display(Markdown(f'### {title} · `{path.name}`'));display(Audio(filename=str(path)))
print('Saídas privadas em',outdir,'; auditar timbre e expressividade antes de integrar.')


### Critérios
- O smoke confirma apenas compatibilidade/memória, não qualidade.
- Compare Base+alegre com Adapter+alegre para isolar efeito do adapter; Adapter+neutro verifica se o controle emocional não colapsou.
- Rejeite adapter que altere o timbre de forma indesejada. Loss não mede emoção, PT-BR ou identidade.
- O helper foi revisado contra o código Qwen pinado, mas ainda precisa de smoke real na T4. Não há garantia de sucesso até ouvir a A/B.
